In [207]:
import pandas as pd 
import numpy as np
import json 
from utils import *
key_test_t=pd.read_csv('./DataInput/key_test_t.csv',on_bad_lines='skip',engine='python').iloc[:,1:]
key_test_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 647974 entries, 0 to 647973
Data columns (total 23 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   id              647974 non-null  object
 1   p_id            647974 non-null  object
 2   results_key     647974 non-null  object
 3   results         647974 non-null  object
 4   run_args        647846 non-null  object
 5   results_files   647974 non-null  object
 6   process_data    647765 non-null  object
 7   baseline        526471 non-null  object
 8   scenes_id       647974 non-null  object
 9   cost_id         647974 non-null  object
 10  log_id          647974 non-null  object
 11  dimension       647974 non-null  object
 12  task_id         647974 non-null  object
 13  run_id          647974 non-null  object
 14  task_from       647974 non-null  object
 15  plan_id         540317 non-null  object
 16  report_id       539425 non-null  object
 17  status          647974 non-nu

In [208]:
key_test_t=key_test_t[["id","results_key","results","scenes_id","dimension","report_id","modified"]]

In [209]:
key_test_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 647974 entries, 0 to 647973
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   id           647974 non-null  object
 1   results_key  647974 non-null  object
 2   results      647974 non-null  object
 3   scenes_id    647974 non-null  object
 4   dimension    647974 non-null  object
 5   report_id    539425 non-null  object
 6   modified     647974 non-null  object
dtypes: object(7)
memory usage: 34.6+ MB


In [210]:
nginx_wrk_index=[]
for i in range(len(key_test_t)):
    dimension=key_test_t["dimension"][i]
    try:
        parse_dimension=json.loads(dimension)
        if(parse_dimension["tool_name"]=="nginx_wrk" and parse_dimension):            
            nginx_wrk_index.append(i)
    except:
        continue

In [211]:
nginx_wrk=key_test_t.iloc[nginx_wrk_index,:].reset_index(drop=True)
nginx_wrk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18104 entries, 0 to 18103
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           18104 non-null  object
 1   results_key  18104 non-null  object
 2   results      18104 non-null  object
 3   scenes_id    18104 non-null  object
 4   dimension    18104 non-null  object
 5   report_id    17318 non-null  object
 6   modified     18104 non-null  object
dtypes: object(7)
memory usage: 990.2+ KB


In [212]:
error_index=[]
for i in range(len(nginx_wrk)):
    result=nginx_wrk["results"][i]
    parse_result=json.loads(result)
    if("wrk_errors_str" in parse_result):
        error_index.append(i)

nginx_wrk_drop_errorlines=nginx_wrk.drop(index=error_index)

In [213]:
nginx_final=nginx_wrk_drop_errorlines.drop_duplicates(subset=["scenes_id","dimension"])
nginx_final.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3353 entries, 0 to 18101
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3353 non-null   object
 1   results_key  3353 non-null   object
 2   results      3353 non-null   object
 3   scenes_id    3353 non-null   object
 4   dimension    3353 non-null   object
 5   report_id    3195 non-null   object
 6   modified     3353 non-null   object
dtypes: object(7)
memory usage: 209.6+ KB


In [214]:
nginx_final_result_keyset=find_all_keys(nginx_final["results"])
nginx_final_result_list=sorted(list(nginx_final_result_keyset))
nginx_final_result_list

['#wrk_qps_avg',
 '#wrk_req_sec_avg',
 'wrk_distribution_90',
 'wrk_latency_avg',
 'wrk_latency_distribution_90',
 'wrk_latency_max',
 'wrk_latency_stdev',
 'wrk_transfer_sec']

In [215]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
nginx_wrk_le=le.fit(nginx_final_result_list)

In [216]:
nginx_yfinal=find_all_value(nginx_final["results"],nginx_wrk_le,nginx_final_result_list)

nginx_yfinal_df=pd.DataFrame(data=nginx_yfinal[1:],columns=nginx_yfinal[0])
nginx_yfinal[0]
nginx_yfinal_df.to_csv('nginx_yfinal.csv')
nginx_yfinal_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3353 entries, 0 to 3352
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   #wrk_qps_avg                 3352 non-null   float64
 1   #wrk_req_sec_avg             1 non-null      float64
 2   wrk_distribution_90          144 non-null    float64
 3   wrk_latency_avg              3353 non-null   float64
 4   wrk_latency_distribution_90  3209 non-null   float64
 5   wrk_latency_max              3353 non-null   float64
 6   wrk_latency_stdev            3353 non-null   float64
 7   wrk_transfer_sec             3352 non-null   float64
dtypes: float64(8)
memory usage: 209.7 KB


In [217]:
nginx_final_dimension_keyset=find_all_keys(nginx_final["dimension"])

In [218]:
nginx_final_dimension_keyset

{'component_name',
 'component_version',
 'cpu_limit',
 'cvm_cpu',
 'cvm_cpu_qos',
 'cvm_cpu_type',
 'cvm_gpu_type',
 'cvm_memory',
 'cvm_os_type',
 'cvm_version',
 'host_configured_clock_speed',
 'host_cpu_isolation:dpdk',
 'host_cpu_isolation:host',
 'host_cpu_isolation:host:dpdk',
 'host_cpu_isolation:host:host',
 'host_cpu_isolation:host:name_8',
 'host_cpu_isolation:host:spdk',
 'host_cpu_isolation:host:vgpu',
 'host_cpu_isolation:host:vm',
 'host_cpu_isolation:name_8',
 'host_cpu_isolation:spdk',
 'host_cpu_isolation:vm',
 'host_cpu_isolation:vrdma',
 'host_cpu_type',
 'host_kernel_version',
 'host_manufacturer_name',
 'host_manufacturer_product_name',
 'host_memory_type',
 'host_nic_type',
 'host_os_version',
 'host_type',
 'k8s_version',
 'kernel_version',
 'mem_limit_mb',
 'name_13',
 'name_1518',
 'name_23',
 'name_731',
 'netPluginType',
 'node_cpu',
 'node_memory',
 'node_nums',
 'node_os_kernel',
 'node_os_name',
 'platform',
 'runtime',
 'runtime_version',
 'test_name',
 

In [219]:
key_name=pd.read_csv('./DataInput/key_name.csv',names=["type","name"])
key_name.dropna(inplace=True)
key_name.reset_index(drop=True,inplace=True)
key_name_dict={}
for i in range(len(key_name)):
   key_name_dict[key_name["name"][i][1:-1]]=key_name["type"][i][1:-1]

In [220]:
nginx_final_dimension_list=list(nginx_final_dimension_keyset)
for index in range(len(nginx_final_dimension_keyset)):
    key=nginx_final_dimension_list[index]
    if(key in key_name_dict):
        nginx_final_dimension_list[index]=key_name_dict[key]
nginx_final_dimension_list=sorted(list(set(nginx_final_dimension_list)))

In [221]:
nginx_wrk_dimension_le=le.fit(nginx_final_dimension_list)

In [222]:
nginx_final_dimension_list

['component_name',
 'component_version',
 'cpu_limit',
 'cvm_cpu',
 'cvm_cpu_qos',
 'cvm_cpu_type',
 'cvm_gpu_type',
 'cvm_memory',
 'cvm_os_type',
 'cvm_version',
 'host_configured_clock_speed',
 'host_cpu_isolation:dpdk',
 'host_cpu_isolation:host',
 'host_cpu_isolation:host:dpdk',
 'host_cpu_isolation:host:host',
 'host_cpu_isolation:host:name_8',
 'host_cpu_isolation:host:spdk',
 'host_cpu_isolation:host:vgpu',
 'host_cpu_isolation:host:vm',
 'host_cpu_isolation:name_8',
 'host_cpu_isolation:spdk',
 'host_cpu_isolation:vm',
 'host_cpu_isolation:vrdma',
 'host_cpu_type',
 'host_kernel_version',
 'host_manufacturer_name',
 'host_manufacturer_product_name',
 'host_memory_type',
 'host_nic_type',
 'host_os_version',
 'host_type',
 'k8s_version',
 'kernel_version',
 'mem_limit_mb',
 'netPluginType',
 'node_cpu',
 'node_memory',
 'node_nums',
 'node_os_kernel',
 'node_os_name',
 'platform',
 'runtime',
 'runtime_version',
 'test_name',
 'tke_cvm_sold_type',
 'tool_name',
 'tool_version']

In [223]:
nginx_wrk_dimension_le.classes_

array(['component_name', 'component_version', 'cpu_limit', 'cvm_cpu',
       'cvm_cpu_qos', 'cvm_cpu_type', 'cvm_gpu_type', 'cvm_memory',
       'cvm_os_type', 'cvm_version', 'host_configured_clock_speed',
       'host_cpu_isolation:dpdk', 'host_cpu_isolation:host',
       'host_cpu_isolation:host:dpdk', 'host_cpu_isolation:host:host',
       'host_cpu_isolation:host:name_8', 'host_cpu_isolation:host:spdk',
       'host_cpu_isolation:host:vgpu', 'host_cpu_isolation:host:vm',
       'host_cpu_isolation:name_8', 'host_cpu_isolation:spdk',
       'host_cpu_isolation:vm', 'host_cpu_isolation:vrdma',
       'host_cpu_type', 'host_kernel_version', 'host_manufacturer_name',
       'host_manufacturer_product_name', 'host_memory_type',
       'host_nic_type', 'host_os_version', 'host_type', 'k8s_version',
       'kernel_version', 'mem_limit_mb', 'netPluginType', 'node_cpu',
       'node_memory', 'node_nums', 'node_os_kernel', 'node_os_name',
       'platform', 'runtime', 'runtime_version', 'tes

In [224]:
nginx_Xfinal=find_dimension_value(nginx_final["dimension"],nginx_wrk_dimension_le,nginx_final_dimension_list,key_name_dict)


In [225]:
nginx_Xfinal_df=pd.DataFrame(data=nginx_Xfinal[1:],columns=nginx_Xfinal[0])
nginx_Xfinal_df.info()
nginx_Xfinal_df["component_name"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3353 entries, 0 to 3352
Data columns (total 47 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   component_name                  3353 non-null   object 
 1   component_version               3353 non-null   object 
 2   cpu_limit                       223 non-null    float64
 3   cvm_cpu                         3106 non-null   object 
 4   cvm_cpu_qos                     2129 non-null   object 
 5   cvm_cpu_type                    2870 non-null   object 
 6   cvm_gpu_type                    44 non-null     object 
 7   cvm_memory                      3106 non-null   object 
 8   cvm_os_type                     3094 non-null   object 
 9   cvm_version                     2832 non-null   object 
 10  host_configured_clock_speed     96 non-null     object 
 11  host_cpu_isolation:dpdk         221 non-null    object 
 12  host_cpu_isolation:host         44

cvm     3127
tke      222
host       4
Name: component_name, dtype: int64

In [226]:
nginx_Xfinal_df["host_cpu_isolation:host:dpdk"].fillna(nginx_Xfinal_df["host_cpu_isolation:host:name_8"],inplace=True)
nginx_Xfinal_df["host_cpu_isolation:dpdk"].fillna(nginx_Xfinal_df["host_cpu_isolation:name_8"],inplace=True)
nginx_Xfinal_df.drop(["host_cpu_isolation:name_8","host_cpu_isolation:host:name_8"],inplace=True,axis=1)
nginx_Xfinal_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3353 entries, 0 to 3352
Data columns (total 45 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   component_name                  3353 non-null   object 
 1   component_version               3353 non-null   object 
 2   cpu_limit                       223 non-null    float64
 3   cvm_cpu                         3106 non-null   object 
 4   cvm_cpu_qos                     2129 non-null   object 
 5   cvm_cpu_type                    2870 non-null   object 
 6   cvm_gpu_type                    44 non-null     object 
 7   cvm_memory                      3106 non-null   object 
 8   cvm_os_type                     3094 non-null   object 
 9   cvm_version                     2832 non-null   object 
 10  host_configured_clock_speed     96 non-null     object 
 11  host_cpu_isolation:dpdk         449 non-null    object 
 12  host_cpu_isolation:host         44

In [231]:
results_key=nginx_final["results_key"]
results_key.reset_index(drop=True,inplace=True)

In [232]:
pd.concat([nginx_Xfinal_df,results_key],axis=1)

,component_name,component_version,cpu_limit,cvm_cpu,cvm_cpu_qos,cvm_cpu_type,cvm_gpu_type,cvm_memory,cvm_os_type,cvm_version,...,node_os_kernel,node_os_name,platform,runtime,runtime_version,test_name,tke_cvm_sold_type,tool_name,tool_version,results_key
0,cvm,default,NaN,62,True,name_0,NaN,214 GB,CentOS Linux release 8.2.2004 (Core),name_1,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_tune_baseline,NaN,nginx_wrk,1.16.7_4.1.0,test_1kb
1,cvm,default,NaN,62,True,name_0,NaN,214 GB,CentOS Linux release 8.2.2004 (Core),name_1,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_accept_mutex_tune_baseline,NaN,nginx_wrk,1.16.4_4.1.0,test_only_string_return
2,tke,debug,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.14.105-19-0020.1.bs,Tencent Linux release 2.4 (Final),NaN,docker,19.3.9,nginx_wrk_tune_baseline,S5,nginx_wrk,1.16.7_4.1.0,test_1kb
3,cvm,default,NaN,62,True,name_0,NaN,212 GB,CentOS Linux release 8.2.2004 (Core),name_17,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_tune_baseline,NaN,nginx_wrk,1.16.7_4.1.0,test_1kb[keep_alive=False]
4,tke,debug,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.14.105-19-0020.1.bs,Tencent Linux release 2.4 (Final),NaN,docker,19.3.9,nginx_wrk_monitor_baseline,S5,nginx_wrk,1.16.7_4.1.0,"test_only_string_return[keep_alive=True,net_pa..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,cvm,default,NaN,32,True,name_0,NaN,106 GB,CentOS Linux release 8.2.2004 (Core),name_17,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_tune_baseline,NaN,nginx_wrk,1.16.7_4.1.0,test_1kb[keep_alive=False]
3349,cvm,default,NaN,32,True,name_0,NaN,106 GB,CentOS Linux release 8.2.2004 (Core),name_17,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_accept_mutex_tune_baseline,NaN,nginx_wrk,1.16.4_4.1.0,test_only_string_return
3350,cvm,default,NaN,16,NaN,name_44,NaN,16 GB,CentOS Linux release 8.2.2004 (Core),name_87,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_accept_mutex_baseline,NaN,nginx_wrk,1.16.4_4.1.0,test_only_string_return
3351,cvm,default,NaN,16,NaN,name_44,NaN,16 GB,CentOS Linux release 8.2.2004 (Core),name_87,...,NaN,NaN,qcloud,NaN,NaN,nginx_wrk_baseline,NaN,nginx_wrk,1.16.7_4.1.0,test_1kb[keep_alive=False]


In [233]:
nginx_Xfinal_df=pd.concat([nginx_Xfinal_df,nginx_final["results_key"]],axis=1)
nginx_Xfinal_df.to_csv('nginx_Xfinal.csv')
nginx_Xfinal_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3353 entries, 0 to 3352
Data columns (total 46 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   component_name                  3353 non-null   object 
 1   component_version               3353 non-null   object 
 2   cpu_limit                       223 non-null    float64
 3   cvm_cpu                         3106 non-null   object 
 4   cvm_cpu_qos                     2129 non-null   object 
 5   cvm_cpu_type                    2870 non-null   object 
 6   cvm_gpu_type                    44 non-null     object 
 7   cvm_memory                      3106 non-null   object 
 8   cvm_os_type                     3094 non-null   object 
 9   cvm_version                     2832 non-null   object 
 10  host_configured_clock_speed     96 non-null     object 
 11  host_cpu_isolation:dpdk         449 non-null    object 
 12  host_cpu_isolation:host         44